In [28]:
# %% [markdown]
# ## 1. Install & Import

# %%
!pip -q install datasets scikit-learn

import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from datasets import load_dataset
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

sns.set_theme(style="whitegrid")

# %% [markdown]
# ## 2. Load Dataset + Bangun Tabel Fitur per Game

# %%
print("Memuat dataset...")

dataset = load_dataset(
    "angeluriot/chess_games",
    split="train",
    streaming=True
)

MAX_GAMES = 15000
MIN_MOVES = 10
MAX_PLY_FOR_HEATMAP = 60   # batas ply dipakai utk heatmap kotak tujuan (biar relevan/cepat)
VALID_END_TYPES = {"checkmate", "resignation", "draw_agreement"}

square_re = re.compile(r"([a-h][1-8])")
files_ = "abcdefgh"
board_counts = np.zeros((8, 8))

rows = []
game_index = 0

for game in dataset:

    if game_index >= MAX_GAMES:
        break

    game_index += 1

    moves = game.get("moves_san")
    white_elo = game.get("white_elo")
    black_elo = game.get("black_elo")
    end_type = game.get("end_type")

    if not moves or len(moves) < MIN_MOVES:
        continue
    if white_elo is None or black_elo is None:
        continue
    if end_type not in VALID_END_TYPES:
        continue

    white_elo = int(white_elo)
    black_elo = int(black_elo)
    n_moves = len(moves)
    n_captures = sum(1 for m in moves if "x" in m)
    n_checks = sum(1 for m in moves if ("+" in m or "#" in m))

    rows.append({
        "white_elo": white_elo,
        "black_elo": black_elo,
        "elo_diff": white_elo - black_elo,
        "avg_elo": (white_elo + black_elo) / 2,
        "n_moves": n_moves,
        "n_captures": n_captures,
        "n_checks": n_checks,
        "end_type": end_type,
    })

    # akumulasi heatmap kotak tujuan (dibatasi MAX_PLY_FOR_HEATMAP ply pertama)
    for tok in moves[:MAX_PLY_FOR_HEATMAP]:
        if tok in ("O-O", "O-O-O"):
            continue
        m = square_re.findall(tok)
        if m:
            dest = m[-1]
            f_idx = files_.index(dest[0])
            r_idx = int(dest[1]) - 1
            board_counts[r_idx, f_idx] += 1

df = pd.DataFrame(rows)

print("Game dibaca (sebelum filter):", game_index)
print("Jumlah game valid dipakai   :", len(df))
df.describe().round(1)

# %% [markdown]
# ## 3. Heatmap #1 — Korelasi Antar Fitur Numerik
#
# Menunjukkan hubungan antar `white_elo`, `black_elo`, `elo_diff`, `avg_elo`, `n_moves`, `n_captures`, `n_checks`.

# %%
numeric_cols = ["white_elo", "black_elo", "elo_diff", "avg_elo", "n_moves", "n_captures", "n_checks"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(7.5, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title("Heatmap Korelasi Fitur Numerik (angeluriot/chess_games)")
plt.tight_layout()
plt.savefig("heatmap_korelasi.png", dpi=150)
plt.show()

# %% [markdown]
# ## 4. Heatmap #2 — Kotak Tujuan Langkah di Papan
#
# Kotak mana yang paling sering jadi tujuan langkah, dihitung dari seluruh game valid (dibatasi 60 ply pertama per game).

# %%
plt.figure(figsize=(7, 6.5))
sns.heatmap(board_counts[::-1], cmap="YlOrRd", annot=True, fmt=".0f",
            xticklabels=list(files_), yticklabels=list("87654321"),
            cbar_kws={"label": "Frekuensi kotak tujuan langkah"})
plt.title(f"Heatmap Papan Catur — Kotak Tujuan Tersering\n({len(df):,} game, {MAX_PLY_FOR_HEATMAP} ply pertama)".replace(",", "."))
plt.xlabel("File")
plt.ylabel("Rank")
plt.tight_layout()
plt.savefig("heatmap_papan_catur.png", dpi=150)
plt.show()

# %% [markdown]
# ## 5. Elbow Method
#
# Cluster game berdasarkan fitur: `white_elo`, `black_elo`, `n_moves`, `n_captures`, `n_checks` (distandarisasi dulu via `StandardScaler`). Inertia (WCSS) dihitung untuk `k = 1..10`.

# %%
CLUSTER_FEATURES = ["white_elo", "black_elo", "n_moves", "n_captures", "n_checks"]

X = df[CLUSTER_FEATURES].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

K_RANGE = range(1, 11)
inertias = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7.5, 5))
plt.plot(list(K_RANGE), inertias, marker="o", linewidth=2, color="#c9974c")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Method — Menentukan Jumlah Cluster Optimal")
plt.xticks(list(K_RANGE))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("elbow_method.png", dpi=150)
plt.show()

print("Inertia per k:")
for k, i in zip(K_RANGE, inertias):
    print(f"  k={k}: {i:.1f}")

# %% [markdown]
# ## 6. Silhouette Score Rata-Rata per k
#
# Melengkapi elbow method — silhouette score mengukur seberapa rapat sebuah titik ke cluster-nya sendiri dibanding cluster tetangga terdekat (rentang -1 s.d. 1, makin tinggi makin baik). Dihitung untuk `k = 2..10` (silhouette tidak terdefinisi untuk `k=1`).

# %%
SIL_K_RANGE = range(2, 11)
silhouette_avgs = []

# silhouette_score bisa berat untuk data besar -> subsample kalau perlu
SAMPLE_FOR_SIL = min(5000, len(X_scaled))
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_scaled), size=SAMPLE_FOR_SIL, replace=False)
X_sil = X_scaled[sample_idx]

for k in SIL_K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sil)
    score = silhouette_score(X_sil, labels)
    silhouette_avgs.append(score)
    print(f"k={k}: silhouette rata-rata = {score:.4f}")

plt.figure(figsize=(7.5, 5))
plt.plot(list(SIL_K_RANGE), silhouette_avgs, marker="o", linewidth=2, color="#7fa896")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("Silhouette Score Rata-Rata")
plt.title(f"Silhouette Score vs k (sampel {SAMPLE_FOR_SIL:,} game)".replace(",", "."))
plt.xticks(list(SIL_K_RANGE))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("silhouette_score_vs_k.png", dpi=150)
plt.show()

best_k = list(SIL_K_RANGE)[int(np.argmax(silhouette_avgs))]
print(f"\nk dengan silhouette tertinggi: {best_k}")

# %% [markdown]
# ## 7. Diagnostic Plot Silhouette Lengkap (per k)
#
# Untuk beberapa nilai `k`, tampilkan sisi-berdampingan:
# - **Kiri**: silhouette plot (tiap "pisau" = satu cluster, lebar = jumlah anggota, panjang tiap garis = nilai silhouette tiap sampel, garis putus-putus merah = rata-rata keseluruhan)
# - **Kanan**: visualisasi cluster pada 2 fitur (`white_elo` vs `n_moves`) diwarnai per cluster
#
# Ini gaya diagnostic plot standar scikit-learn untuk membandingkan kualitas clustering antar `k`.

# %%
K_TO_INSPECT = [2, 3, 4, 5]

for k in K_TO_INSPECT:

    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = km.fit_predict(X_sil)

    sil_avg = silhouette_score(X_sil, cluster_labels)
    sample_sil_values = silhouette_samples(X_sil, cluster_labels)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    # --- Panel kiri: silhouette plot ---
    y_lower = 10
    colors_map = cm.get_cmap("Spectral")(np.linspace(0, 1, k))

    for i in range(k):
        ith_vals = sample_sil_values[cluster_labels == i]
        ith_vals.sort()
        size_i = ith_vals.shape[0]
        y_upper = y_lower + size_i

        ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_vals,
                           facecolor=colors_map[i], edgecolor=colors_map[i], alpha=0.75)
        ax1.text(-0.02, y_lower + 0.5 * size_i, str(i))
        y_lower = y_upper + 10

    ax1.axvline(x=sil_avg, color="red", linestyle="--", label=f"rata-rata = {sil_avg:.3f}")
    ax1.set_title(f"Silhouette Plot (k={k})")
    ax1.set_xlabel("Nilai Koefisien Silhouette")
    ax1.set_ylabel("Label Cluster")
    ax1.set_yticks([])
    ax1.set_xlim(-0.2, 1)
    ax1.legend(loc="upper right")

    # --- Panel kanan: visualisasi cluster (2 fitur asli) ---
    feat_x_idx = CLUSTER_FEATURES.index("white_elo")
    feat_y_idx = CLUSTER_FEATURES.index("n_moves")

    ax2.scatter(X_sil[:, feat_x_idx], X_sil[:, feat_y_idx],
                c=cluster_labels, cmap="Spectral", alpha=0.6, s=18)

    centers_orig = scaler.inverse_transform(km.cluster_centers_)
    ax2.scatter(centers_orig[:, feat_x_idx], centers_orig[:, feat_y_idx],
                marker="X", s=200, c="black", label="centroid")

    ax2.set_title(f"Visualisasi Cluster (k={k})")
    ax2.set_xlabel("white_elo (standarisasi)")
    ax2.set_ylabel("n_moves (standarisasi)")
    ax2.legend()

    plt.suptitle(f"Diagnostic Silhouette — k={k}  |  Skor rata-rata = {sil_avg:.4f}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"silhouette_detail_k{k}.png", dpi=150)
    plt.show()

# %% [markdown]
# ## 8. Profil Cluster Final
#
# Pakai `k` dengan silhouette terbaik (dari section 6), lalu lihat karakteristik tiap cluster pada data penuh (bukan subsample).

# %%
FINAL_K = best_k   # ganti manual di sini kalau mau override, mis. FINAL_K = 4

km_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_scaled)

print(f"Profil cluster (k={FINAL_K}), rata-rata tiap fitur:")
display(df.groupby("cluster")[CLUSTER_FEATURES].mean().round(1))

print("\nJumlah game per cluster:")
display(df["cluster"].value_counts().sort_index())


Memuat dataset...
Game dibaca (sebelum filter): 15000
Jumlah game valid dipakai   : 12702
Inertia per k:
  k=1: 63510.0
  k=2: 43931.5
  k=3: 31645.4
  k=4: 26868.2
  k=5: 23415.0
  k=6: 20838.4
  k=7: 18940.1
  k=8: 17524.7
  k=9: 16458.7
  k=10: 15622.9
k=2: silhouette rata-rata = 0.2944
k=3: silhouette rata-rata = 0.3126
k=4: silhouette rata-rata = 0.2727
k=5: silhouette rata-rata = 0.2670
k=6: silhouette rata-rata = 0.2440
k=7: silhouette rata-rata = 0.2469
k=8: silhouette rata-rata = 0.2388
k=9: silhouette rata-rata = 0.2337
k=10: silhouette rata-rata = 0.2288

k dengan silhouette tertinggi: 3


/tmp/ipykernel_947/3919264516.py:242: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
/tmp/ipykernel_947/3919264516.py:246: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors_map = cm.get_cmap("Spectral")(np.linspace(0, 1, k))
/tmp/ipykernel_947/3919264516.py:246: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors_m

Profil cluster (k=3), rata-rata tiap fitur:


,white_elo,black_elo,n_moves,n_captures,n_checks
cluster,,,,,
0,2531.8,2531.7,126.9,24.4,10.6
1,2543.2,2541.7,63.2,13.6,2.5
2,2054.1,2046.9,76.7,16.1,3.5



Jumlah game per cluster:


,count
cluster,
0,3460
1,5455
2,3787


In [29]:
# %% [markdown]
# Chess Analytics — Heatmap, Elbow, Silhouette — versi .py untuk Google Colab
#
# Cara pakai: upload file .py ini langsung ke Colab (File > Upload notebook),
# atau copy-paste tiap blok '# %%' ke cell baru satu per satu.

# %% [markdown]
# #  Analisis Statistik Dataset Catur — Heatmap, Elbow, & Silhouette (Lengkap)
#
# Sumber data: [`angeluriot/chess_games`](https://github.com/angeluriot/Chess_games) (Hugging Face), sama seperti yang dipakai untuk training model RandomForest sebelumnya.
#
# Notebook ini fokus ke tiga analisis:
#
# 1. **Heatmap** — korelasi antar fitur numerik game, dan heatmap kotak tujuan langkah di papan
# 2. **Elbow Method** — inertia (WCSS) vs jumlah cluster `k`, untuk menentukan `k` yang masuk akal
# 3. **Silhouette Analysis (lengkap)** — skor silhouette rata-rata per `k`, **plus** diagnostic plot detail (silhouette per sampel + visualisasi cluster) untuk beberapa nilai `k`, gaya standar scikit-learn
#
# Clustering dilakukan pada level **game** (bukan per-langkah), memakai fitur: elo putih, elo hitam, jumlah langkah, jumlah capture, dan jumlah skak per game.

# %% [markdown]
# ## 1. Install & Import

# %%
!pip -q install datasets scikit-learn

import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from datasets import load_dataset
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

sns.set_theme(style="whitegrid")


# %% [markdown]
# ## 2. Load Dataset + Bangun Tabel Fitur per Game

# %%
print("Memuat dataset...")

dataset = load_dataset(
    "angeluriot/chess_games",
    split="train",
    streaming=True
)

MAX_GAMES = 15000
MIN_MOVES = 10
MAX_PLY_FOR_HEATMAP = 60   # batas ply dipakai utk heatmap kotak tujuan (biar relevan/cepat)
VALID_END_TYPES = {"checkmate", "resignation", "draw_agreement"}

square_re = re.compile(r"([a-h][1-8])")
files_ = "abcdefgh"
board_counts = np.zeros((8, 8))

rows = []
game_index = 0
n_skipped_null_elo = 0
n_skipped_bad_moves = 0
n_skipped_end_type = 0

for game in dataset:

    if game_index >= MAX_GAMES:
        break

    game_index += 1

    moves = game.get("moves_san")
    white_elo = game.get("white_elo")
    black_elo = game.get("black_elo")
    end_type = game.get("end_type")

    if not moves or len(moves) < MIN_MOVES:
        n_skipped_bad_moves += 1
        continue

    # PENTING: elo bisa null/None ATAU NaN tergantung backend datasets library --
    # pd.isna() menangkap None, NaN, dan nilai kosong lain sekaligus (lebih aman
    # daripada cuma "is None", yang bisa lolos kalau nilainya float('nan')).
    if white_elo is None or black_elo is None or pd.isna(white_elo) or pd.isna(black_elo):
        n_skipped_null_elo += 1
        continue

    if end_type not in VALID_END_TYPES:
        n_skipped_end_type += 1
        continue

    try:
        white_elo = int(white_elo)
        black_elo = int(black_elo)
    except (TypeError, ValueError):
        n_skipped_null_elo += 1
        continue

    n_moves = len(moves)
    n_captures = sum(1 for m in moves if "x" in m)
    n_checks = sum(1 for m in moves if ("+" in m or "#" in m))

    rows.append({
        "white_elo": white_elo,
        "black_elo": black_elo,
        "elo_diff": white_elo - black_elo,
        "avg_elo": (white_elo + black_elo) / 2,
        "n_moves": n_moves,
        "n_captures": n_captures,
        "n_checks": n_checks,
        "end_type": end_type,
    })

    # akumulasi heatmap kotak tujuan (dibatasi MAX_PLY_FOR_HEATMAP ply pertama)
    for tok in moves[:MAX_PLY_FOR_HEATMAP]:
        if tok in ("O-O", "O-O-O"):
            continue
        m = square_re.findall(tok)
        if m:
            dest = m[-1]
            f_idx = files_.index(dest[0])
            r_idx = int(dest[1]) - 1
            board_counts[r_idx, f_idx] += 1

df = pd.DataFrame(rows)

# safety net tambahan: buang baris yang masih ada NaN di kolom numerik apa pun
# (mis. kalau ada kolom turunan yang tak sengaja jadi NaN)
numeric_cols_all = ["white_elo", "black_elo", "elo_diff", "avg_elo", "n_moves", "n_captures", "n_checks"]
before_dropna = len(df)
df = df.dropna(subset=numeric_cols_all).reset_index(drop=True)
after_dropna = len(df)

print("Game dibaca (sebelum filter)   :", game_index)
print("Dilewati - elo null/tidak valid:", n_skipped_null_elo)
print("Dilewati - moves kurang/kosong :", n_skipped_bad_moves)
print("Dilewati - end_type tak masuk  :", n_skipped_end_type)
print("Baris dibuang krn NaN residual :", before_dropna - after_dropna)
print("Jumlah game valid dipakai      :", len(df))

assert len(df) > 0, (
    "Tidak ada game valid tersisa! Coba naikkan MAX_GAMES, atau longgarkan "
    "VALID_END_TYPES / MIN_MOVES di atas."
)

df.describe().round(1)


# %% [markdown]
# ## 3. Heatmap #1 — Korelasi Antar Fitur Numerik
#
# Menunjukkan hubungan antar `white_elo`, `black_elo`, `elo_diff`, `avg_elo`, `n_moves`, `n_captures`, `n_checks`.

# %%
numeric_cols = ["white_elo", "black_elo", "elo_diff", "avg_elo", "n_moves", "n_captures", "n_checks"]

# guard: buang kolom yang variance-nya 0 (mis. semua nilainya sama persis) --
# korelasi dengan kolom seperti itu otomatis NaN (0/0) dan bikin heatmap
# penuh sel kosong/putih.
std_check = df[numeric_cols].std()
zero_var_cols = std_check[std_check == 0].index.tolist()
if zero_var_cols:
    print("Kolom dengan variance 0 (dibuang dari heatmap):", zero_var_cols)
usable_cols = [c for c in numeric_cols if c not in zero_var_cols]

corr = df[usable_cols].corr()

# guard tambahan: kalau masih ada NaN residual (mis. akibat sampel terlalu
# sedikit), tampilkan sebagai 0 di heatmap supaya tidak error/blank total,
# tapi beri tahu di console.
n_nan_cells = corr.isna().sum().sum()
if n_nan_cells > 0:
    print(f"Peringatan: {n_nan_cells} sel korelasi NaN, ditampilkan sebagai 0.")
    corr = corr.fillna(0)

plt.figure(figsize=(7.5, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title("Heatmap Korelasi Fitur Numerik (angeluriot/chess_games)")
plt.tight_layout()
plt.savefig("heatmap_korelasi.png", dpi=150)
plt.show()


# %% [markdown]
# ## 4. Heatmap #2 — Kotak Tujuan Langkah di Papan
#
# Kotak mana yang paling sering jadi tujuan langkah, dihitung dari seluruh game valid (dibatasi 60 ply pertama per game).

# %%
assert board_counts.sum() > 0, (
    "board_counts semuanya nol -- kemungkinan tidak ada game valid yang "
    "lolos filter di Section 2. Cek pesan 'Jumlah game valid dipakai' di atas."
)

plt.figure(figsize=(7, 6.5))
sns.heatmap(board_counts[::-1], cmap="YlOrRd", annot=True, fmt=".0f",
            xticklabels=list(files_), yticklabels=list("87654321"),
            cbar_kws={"label": "Frekuensi kotak tujuan langkah"})
plt.title(f"Heatmap Papan Catur — Kotak Tujuan Tersering\n({len(df):,} game, {MAX_PLY_FOR_HEATMAP} ply pertama)".replace(",", "."))
plt.xlabel("File")
plt.ylabel("Rank")
plt.tight_layout()
plt.savefig("heatmap_papan_catur.png", dpi=150)
plt.show()


# %% [markdown]
# ## 5. Elbow Method
#
# Cluster game berdasarkan fitur: `white_elo`, `black_elo`, `n_moves`, `n_captures`, `n_checks` (distandarisasi dulu via `StandardScaler`). Inertia (WCSS) dihitung untuk `k = 1..10`.

# %%
CLUSTER_FEATURES = ["white_elo", "black_elo", "n_moves", "n_captures", "n_checks"]

X = df[CLUSTER_FEATURES].copy()

# guard terakhir sebelum clustering: KMeans/StandardScaler akan error keras
# ("Input X contains NaN") kalau masih ada NaN yang lolos dari Section 2.
n_nan = X.isna().sum().sum()
if n_nan > 0:
    print(f"Peringatan: {n_nan} nilai NaN ditemukan di fitur cluster, baris terkait dibuang.")
    valid_mask = ~X.isna().any(axis=1)
    X = X[valid_mask].reset_index(drop=True)
    df = df[valid_mask].reset_index(drop=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

K_RANGE = range(1, 11)
inertias = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7.5, 5))
plt.plot(list(K_RANGE), inertias, marker="o", linewidth=2, color="#c9974c")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Method — Menentukan Jumlah Cluster Optimal")
plt.xticks(list(K_RANGE))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("elbow_method.png", dpi=150)
plt.show()

print("Inertia per k:")
for k, i in zip(K_RANGE, inertias):
    print(f"  k={k}: {i:.1f}")


# %% [markdown]
# ## 6. Silhouette Score Rata-Rata per k
#
# Melengkapi elbow method — silhouette score mengukur seberapa rapat sebuah titik ke cluster-nya sendiri dibanding cluster tetangga terdekat (rentang -1 s.d. 1, makin tinggi makin baik). Dihitung untuk `k = 2..10` (silhouette tidak terdefinisi untuk `k=1`).

# %%
SIL_K_RANGE = range(2, 11)
silhouette_avgs = []

# silhouette_score bisa berat untuk data besar -> subsample kalau perlu
SAMPLE_FOR_SIL = min(5000, len(X_scaled))
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_scaled), size=SAMPLE_FOR_SIL, replace=False)
X_sil = X_scaled[sample_idx]

for k in SIL_K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sil)
    score = silhouette_score(X_sil, labels)
    silhouette_avgs.append(score)
    print(f"k={k}: silhouette rata-rata = {score:.4f}")

plt.figure(figsize=(7.5, 5))
plt.plot(list(SIL_K_RANGE), silhouette_avgs, marker="o", linewidth=2, color="#7fa896")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("Silhouette Score Rata-Rata")
plt.title(f"Silhouette Score vs k (sampel {SAMPLE_FOR_SIL:,} game)".replace(",", "."))
plt.xticks(list(SIL_K_RANGE))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("silhouette_score_vs_k.png", dpi=150)
plt.show()

best_k = list(SIL_K_RANGE)[int(np.argmax(silhouette_avgs))]
print(f"\nk dengan silhouette tertinggi: {best_k}")


# %% [markdown]
# ## 7. Diagnostic Plot Silhouette Lengkap (per k)
#
# Untuk beberapa nilai `k`, tampilkan sisi-berdampingan:
# - **Kiri**: silhouette plot (tiap "pisau" = satu cluster, lebar = jumlah anggota, panjang tiap garis = nilai silhouette tiap sampel, garis putus-putus merah = rata-rata keseluruhan)
# - **Kanan**: visualisasi cluster pada 2 fitur (`white_elo` vs `n_moves`) diwarnai per cluster
#
# Ini gaya diagnostic plot standar scikit-learn untuk membandingkan kualitas clustering antar `k`.

# %%
K_TO_INSPECT = [2, 3, 4, 5]

for k in K_TO_INSPECT:

    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = km.fit_predict(X_sil)

    sil_avg = silhouette_score(X_sil, cluster_labels)
    sample_sil_values = silhouette_samples(X_sil, cluster_labels)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    # --- Panel kiri: silhouette plot ---
    y_lower = 10
    colors_map = cm.get_cmap("Spectral")(np.linspace(0, 1, k))

    for i in range(k):
        ith_vals = sample_sil_values[cluster_labels == i]
        ith_vals.sort()
        size_i = ith_vals.shape[0]
        y_upper = y_lower + size_i

        ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_vals,
                           facecolor=colors_map[i], edgecolor=colors_map[i], alpha=0.75)
        ax1.text(-0.02, y_lower + 0.5 * size_i, str(i))
        y_lower = y_upper + 10

    ax1.axvline(x=sil_avg, color="red", linestyle="--", label=f"rata-rata = {sil_avg:.3f}")
    ax1.set_title(f"Silhouette Plot (k={k})")
    ax1.set_xlabel("Nilai Koefisien Silhouette")
    ax1.set_ylabel("Label Cluster")
    ax1.set_yticks([])
    ax1.set_xlim(-0.2, 1)
    ax1.legend(loc="upper right")

    # --- Panel kanan: visualisasi cluster (2 fitur asli) ---
    feat_x_idx = CLUSTER_FEATURES.index("white_elo")
    feat_y_idx = CLUSTER_FEATURES.index("n_moves")

    ax2.scatter(X_sil[:, feat_x_idx], X_sil[:, feat_y_idx],
                c=cluster_labels, cmap="Spectral", alpha=0.6, s=18)

    centers_orig = scaler.inverse_transform(km.cluster_centers_)
    ax2.scatter(centers_orig[:, feat_x_idx], centers_orig[:, feat_y_idx],
                marker="X", s=200, c="black", label="centroid")

    ax2.set_title(f"Visualisasi Cluster (k={k})")
    ax2.set_xlabel("white_elo (standarisasi)")
    ax2.set_ylabel("n_moves (standarisasi)")
    ax2.legend()

    plt.suptitle(f"Diagnostic Silhouette — k={k}  |  Skor rata-rata = {sil_avg:.4f}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"silhouette_detail_k{k}.png", dpi=150)
    plt.show()


# %% [markdown]
# ## 8. Profil Cluster Final
#
# Pakai `k` dengan silhouette terbaik (dari section 6), lalu lihat karakteristik tiap cluster pada data penuh (bukan subsample).

# %%
FINAL_K = best_k   # ganti manual di sini kalau mau override, mis. FINAL_K = 4

km_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_scaled)

print(f"Profil cluster (k={FINAL_K}), rata-rata tiap fitur:")
display(df.groupby("cluster")[CLUSTER_FEATURES].mean().round(1))

print("\nJumlah game per cluster:")
display(df["cluster"].value_counts().sort_index())



Memuat dataset...
Game dibaca (sebelum filter)   : 15000
Dilewati - elo null/tidak valid: 2250
Dilewati - moves kurang/kosong : 0
Dilewati - end_type tak masuk  : 48
Baris dibuang krn NaN residual : 0
Jumlah game valid dipakai      : 12702
Inertia per k:
  k=1: 63510.0
  k=2: 43931.5
  k=3: 31645.4
  k=4: 26868.2
  k=5: 23415.0
  k=6: 20838.4
  k=7: 18940.1
  k=8: 17524.7
  k=9: 16458.7
  k=10: 15622.9
k=2: silhouette rata-rata = 0.2944
k=3: silhouette rata-rata = 0.3126
k=4: silhouette rata-rata = 0.2727
k=5: silhouette rata-rata = 0.2670
k=6: silhouette rata-rata = 0.2440
k=7: silhouette rata-rata = 0.2469
k=8: silhouette rata-rata = 0.2388
k=9: silhouette rata-rata = 0.2337
k=10: silhouette rata-rata = 0.2288

k dengan silhouette tertinggi: 3


/tmp/ipykernel_947/2549647955.py:318: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors_map = cm.get_cmap("Spectral")(np.linspace(0, 1, k))
/tmp/ipykernel_947/2549647955.py:318: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors_map = cm.get_cmap("Spectral")(np.linspace(0, 1, k))
/tmp/ipykernel_947/2549647955.py:318: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors_map = cm.get_cmap("Spectral")(np.linspace(0, 1, k))
/tmp/ipykernel_947/2549647955.py:318:

Profil cluster (k=3), rata-rata tiap fitur:


,white_elo,black_elo,n_moves,n_captures,n_checks
cluster,,,,,
0,2531.8,2531.7,126.9,24.4,10.6
1,2543.2,2541.7,63.2,13.6,2.5
2,2054.1,2046.9,76.7,16.1,3.5



Jumlah game per cluster:


,count
cluster,
0,3460
1,5455
2,3787


In [31]:
# %% [markdown]
# Chess AI RandomForest + Papan Interaktif — versi .py untuk Google Colab
#
# Cara pakai:
# 1. Buka https://colab.research.google.com
# 2. File > Upload notebook > pilih file .py ini (Colab otomatis
#    memecahnya jadi cell-cell terpisah karena tanda '# %%')
#    ATAU copy-paste tiap blok '# %%' ke cell baru secara manual.
# 3. Jalankan cell dari atas ke bawah (Runtime > Run all).

# %% [markdown]
# # ♟️ Real-Time Chess AI — RandomForest + Papan Interaktif Terintegrasi
#
# Notebook ini menggabungkan:
# - **Cell 1–8**: kode training model (RandomForest, fitur one-hot 3-langkah + ply + elo, split per-game) — persis seperti yang sudah dijalankan sebelumnya.
# - **Cell 9 dst (BARU)**: papan catur interaktif (`python-chess` + `ipywidgets`) di mana **setiap giliran, prediksi model ML ditampilkan langsung di papan** — kotak tujuan langkah favorit ditandai lingkaran, dan panel di samping menunjukkan ranking top-5 kandidat legal beserta probabilitasnya.
#
# Prediksi otomatis menyesuaikan diri dengan elo yang Anda atur (widget slider), karena model memang dilatih memakai `white_elo`, `black_elo`, `elo_diff` sebagai fitur.

# %% [markdown]
# ## 1–2. Install & Import

# %%
!pip -q install python-chess ipywidgets datasets scikit-learn

import pandas as pd
import numpy as np
import chess
import chess.svg
import ipywidgets as widgets

from datasets import load_dataset
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, top_k_accuracy_score
import scipy.sparse as sp

from IPython.display import display, HTML

# %% [markdown]
# ## 3. Load Dataset + Filter Kualitas

# %%
print("Memuat dataset...")

dataset = load_dataset(
    "angeluriot/chess_games",
    split="train",
    streaming=True
)

MAX_GAMES = 15000
MIN_MOVES = 10
MAX_PLY_PER_GAME = 30
WINDOW_STRIDE = 1
VALID_END_TYPES = {"checkmate", "resignation", "draw_agreement"}

rows = []
game_index = 0

for game in dataset:

    if game_index >= MAX_GAMES:
        break

    game_index += 1

    moves = game.get("moves_san")
    white_elo = game.get("white_elo")
    black_elo = game.get("black_elo")
    end_type = game.get("end_type")

    if not moves or len(moves) < MIN_MOVES:
        continue
    if white_elo is None or black_elo is None:
        continue
    if end_type not in VALID_END_TYPES:
        continue

    white_elo = int(white_elo)
    black_elo = int(black_elo)
    elo_diff = white_elo - black_elo

    limit = min(len(moves) - 3, MAX_PLY_PER_GAME)

    for i in range(0, limit, WINDOW_STRIDE):
        rows.append({
            "game_id": game_index,
            "ply": i,
            "m3": moves[i],
            "m2": moves[i + 1],
            "m1": moves[i + 2],
            "white_elo": white_elo,
            "black_elo": black_elo,
            "elo_diff": elo_diff,
            "target": moves[i + 3],
        })

print("Game dibaca (sebelum filter):", game_index)
print("Jumlah window sequence      :", len(rows))

df_moves = pd.DataFrame(rows)
print("Jumlah game valid setelah filter:", df_moves["game_id"].nunique())

# %% [markdown]
# ## 4. Ambil Target Paling Sering Muncul

# %%
TOP_TARGETS = 80

top_targets = (
    df_moves["target"]
    .value_counts()
    .nlargest(TOP_TARGETS)
    .index
)

df_filtered = df_moves[df_moves["target"].isin(top_targets)].copy()

print("\nJumlah data setelah filtering :", len(df_filtered))
print("Jumlah kelas target           :", df_filtered["target"].nunique())
print("Cakupan target terhadap semua kemunculan: "
      f"{len(df_filtered) / len(df_moves) * 100:.1f}%")

# %% [markdown]
# ## 5. Feature Engineering

# %%
move_ohe = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    min_frequency=5
)

move_cols = df_filtered[["m3", "m2", "m1"]]
X_moves = move_ohe.fit_transform(move_cols)

NUMERIC_COLS = ["ply", "white_elo", "black_elo", "elo_diff"]
X_numeric = df_filtered[NUMERIC_COLS].to_numpy()
X = sp.hstack([X_moves, sp.csr_matrix(X_numeric)]).tocsr()

y = df_filtered["target"].to_numpy()
groups = df_filtered["game_id"].to_numpy()

print("Ukuran fitur:", X.shape)

# %% [markdown]
# ## 6. Train / Test Split — per Game

# %%
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {X_train.shape[0]} baris | Test: {X_test.shape[0]} baris")
print(f"Game train: {len(set(groups[train_idx]))} | "
      f"Game test: {len(set(groups[test_idx]))} (tidak overlap)")

# %% [markdown]
# ## 7. Random Forest

# %%
print("\nTraining Random Forest...")

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=3,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

print("Training selesai!")

# %% [markdown]
# ## 8. Evaluasi

# %%
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
top3 = top_k_accuracy_score(y_test, y_proba, k=3, labels=model.classes_)
top5 = top_k_accuracy_score(y_test, y_proba, k=5, labels=model.classes_)

print(f"\nAkurasi top-1 : {acc * 100:.2f}%")
print(f"Akurasi top-3 : {top3 * 100:.2f}%")
print(f"Akurasi top-5 : {top5 * 100:.2f}%")

baseline_class = pd.Series(y_train).mode()[0]
baseline_acc = accuracy_score(y_test, [baseline_class] * len(y_test))
print(f"Baseline (tebak kelas mayoritas \'{baseline_class}\'): {baseline_acc * 100:.2f}%")

test_ply = df_filtered["ply"].to_numpy()[test_idx]
print("\nAkurasi per rentang ply:")
for lo, hi in [(0, 6), (6, 12), (12, 20), (20, 30)]:
    mask = (test_ply >= lo) & (test_ply < hi)
    if mask.sum() > 20:
        print(f"  ply {lo}-{hi}: {accuracy_score(y_test[mask], y_pred[mask]) * 100:.2f}% "
              f"(n={mask.sum()})")

n_numeric = X_numeric.shape[1]
numeric_importance = model.feature_importances_[-n_numeric:]
print("\nImportance fitur numerik (ply, white_elo, black_elo, elo_diff):")
for name, imp in zip(NUMERIC_COLS, numeric_importance):
    print(f"  {name}: {imp:.4f}")

# %% [markdown]
# ## 9. Papan Interaktif dengan Prediksi Model Real-Time (BARU)
#
# Board di bawah ini pakai `python-chess` untuk state & legalitas langkah (sama seperti board awal Anda), tapi fungsi prediksinya sekarang benar-benar memanggil **model RandomForest hasil training di atas** — bukan lagi `CountVectorizer`.
#
# **Perbedaan penting dari fungsi prediksi versi lama:**
# - Input model sekarang 3 kolom terpisah (`m3`, `m2`, `m1`) di-one-hot pakai `move_ohe` yang sama persis dipakai saat training (bukan `vectorizer.transform` lagi).
# - Ditambah 4 fitur numerik: `ply` (posisi window context, konsisten dengan definisi saat training: `ply = jumlah_langkah_sudah_dimainkan - 3`), `white_elo`, `black_elo`, `elo_diff`.
# - Elo diatur lewat slider — jadi Anda bisa lihat bagaimana prediksi berubah kalau "berpura-pura" jadi game antara pemain elo 1800 vs pemain elo 2600, misalnya.
# - Kandidat prediksi tetap disaring ke langkah yang benar-benar **legal** di posisi papan saat ini (persis seperti sebelumnya), lalu ditandai langsung di papan dengan lingkaran pada kotak tujuannya.

# %%
# =========================================================
# 9. STATE PAPAN & WIDGET ELO
# =========================================================

board = chess.Board()
move_history = []
selected_square = None

board_output = widgets.Output()
history_output = widgets.Output()
prediction_output = widgets.Output()

white_elo_slider = widgets.IntSlider(
    value=2200, min=800, max=2900, step=50,
    description="Elo Putih:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="320px")
)

black_elo_slider = widgets.IntSlider(
    value=2200, min=800, max=2900, step=50,
    description="Elo Hitam:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="320px")
)

# %%
# =========================================================
# 10. TAMPILKAN BOARD (SVG) + highlight kotak prediksi
# =========================================================

def kotak_prediksi_teratas():
    """Kembalikan set kotak tujuan dari top-3 prediksi legal saat ini,
    dipakai untuk memberi tanda visual di board SVG (fill warna)."""
    hasil = hitung_prediksi()
    fill = {}
    warna = ["#f2c14e88", "#f2c14e55", "#f2c14e33"]
    for idx, (san, prob, chess_move) in enumerate(hasil[:3]):
        fill[chess_move.to_square] = warna[idx] if idx < len(warna) else warna[-1]
    return fill


def tampilkan_board():

    fill_dict = kotak_prediksi_teratas() if len(move_history) >= 3 else {}

    svg = chess.svg.board(
        board=board,
        size=500,
        lastmove=(board.peek() if board.move_stack else None),
        fill=fill_dict
    )

    board_output.clear_output(wait=True)

    with board_output:
        display(HTML(svg))

# %%
# =========================================================
# 11. PREDIKSI REAL-TIME -- SEKARANG MEMAKAI MODEL RANDOM FOREST
# =========================================================

def hitung_prediksi():
    """Kembalikan list (san, probabilitas, chess.Move) terurut dari
    probabilitas tertinggi, hanya untuk langkah yang legal di posisi
    board saat ini. List kosong kalau data belum cukup atau tidak ada
    kandidat model yang match dengan langkah legal."""

    if len(move_history) < 3:
        return []

    m3, m2, m1 = move_history[-3], move_history[-2], move_history[-1]
    ply_feature = len(move_history) - 3   # konsisten dgn definisi ply saat training

    context_df = pd.DataFrame([[m3, m2, m1]], columns=["m3", "m2", "m1"])
    X_moves_input = move_ohe.transform(context_df)

    X_numeric_input = np.array([[
        ply_feature,
        white_elo_slider.value,
        black_elo_slider.value,
        white_elo_slider.value - black_elo_slider.value,
    ]])

    X_input = sp.hstack([X_moves_input, sp.csr_matrix(X_numeric_input)]).tocsr()

    probabilities = model.predict_proba(X_input)[0]
    classes = model.classes_

    legal_moves = list(board.legal_moves)
    legal_san = {}
    for mv in legal_moves:
        try:
            legal_san[board.san(mv)] = mv
        except Exception:
            pass

    kandidat = []
    for i, prob in enumerate(probabilities):
        move_name = classes[i]
        if move_name in legal_san:
            kandidat.append((move_name, prob, legal_san[move_name]))

    kandidat.sort(key=lambda x: x[1], reverse=True)
    return kandidat


def prediksi_realtime():

    prediction_output.clear_output(wait=True)

    if len(move_history) < 3:
        with prediction_output:
            print("Prediksi belum tersedia.")
            print(f"Masukkan minimal 3 langkah. Saat ini: {len(move_history)}/3")
        return

    context = " ".join(move_history[-3:])
    kandidat = hitung_prediksi()

    with prediction_output:

        print("====================================")
        print("   PREDIKSI MACHINE LEARNING (RF)")
        print("====================================")
        print()
        print("3 langkah terakhir :", context)
        print(f"Elo Putih / Hitam  : {white_elo_slider.value} / {black_elo_slider.value}")
        print()

        if not kandidat:
            print("Model tidak menemukan prediksi yang legal pada posisi saat ini.")
            print("(Wajar untuk posisi yang jarang/tidak ada di data training --")
            print(" model ini hanya mengenal", TOP_TARGETS, "jenis langkah target.)")
            return

        best_move, best_prob, _ = kandidat[0]
        print(f"Prediksi terbaik : {best_move}")
        print(f"Probabilitas     : {best_prob * 100:.2f}%")
        print()

        for no, (move_name, prob, _) in enumerate(kandidat[:5], start=1):
            print(f"{no}. {move_name:<8} {prob * 100:.2f}%")

# %%
# =========================================================
# 12. RIWAYAT LANGKAH
# =========================================================

def tampilkan_riwayat():

    history_output.clear_output()

    with history_output:

        if not move_history:
            print("Belum ada langkah.")
            return

        print("Riwayat langkah:")

        for i in range(0, len(move_history), 2):
            nomor = i // 2 + 1
            white_move = move_history[i]

            if i + 1 < len(move_history):
                black_move = move_history[i + 1]
                print(f"{nomor}. {white_move} {black_move}")
            else:
                print(f"{nomor}. {white_move}")

# %%
# =========================================================
# 13. LAKUKAN LANGKAH
# =========================================================

def lakukan_langkah(move):

    try:
        san = board.san(move)
        board.push(move)
        move_history.append(san)

        tampilkan_board()
        tampilkan_riwayat()
        prediksi_realtime()

    except Exception as e:
        print("Error:", e)

# %%
# =========================================================
# 14. SIMBOL BUAH CATUR & PAPAN TOMBOL
# =========================================================

piece_symbols = {
    "P": "♙", "N": "♘", "B": "♗", "R": "♖", "Q": "♕", "K": "♔",
    "p": "♟", "n": "♞", "b": "♝", "r": "♜", "q": "♛", "k": "♚",
}


def buat_papan_tombol():

    buttons = []

    for rank in range(7, -1, -1):
        row = []
        for file in range(8):
            square = chess.square(file, rank)
            button = widgets.Button(
                description=" ",
                layout=widgets.Layout(width="60px", height="60px")
            )
            button.square = square
            button.on_click(lambda b, s=square: klik_square(s))
            row.append(button)
        buttons.append(row)

    return buttons


def update_buttons():

    for row in board_buttons:
        for button in row:
            square = button.square
            piece = board.piece_at(square)
            button.description = piece_symbols[piece.symbol()] if piece else " "

# %%
# =========================================================
# 15. KLIK PETAK
# =========================================================

def klik_square(square):

    global selected_square

    if selected_square is None:
        piece = board.piece_at(square)
        if piece is None:
            return
        if piece.color != board.turn:
            return
        selected_square = square
        return

    move = chess.Move(selected_square, square)

    if move in board.legal_moves:
        lakukan_langkah(move)
        selected_square = None
        update_buttons()
        return

    move_queen = chess.Move(selected_square, square, promotion=chess.QUEEN)
    if move_queen in board.legal_moves:
        lakukan_langkah(move_queen)
        selected_square = None
        update_buttons()
        return

    selected_square = None

# %%
# =========================================================
# 16. RESET / UNDO
# =========================================================

def reset_board(b=None):

    global board, move_history, selected_square

    board = chess.Board()
    move_history = []
    selected_square = None

    update_buttons()
    tampilkan_board()
    tampilkan_riwayat()

    prediction_output.clear_output()
    with prediction_output:
        print("Prediksi akan muncul setelah 3 langkah.")


def undo_move(b=None):

    global selected_square

    if board.move_stack:
        board.pop()
        if move_history:
            move_history.pop()

    selected_square = None

    update_buttons()
    tampilkan_board()
    tampilkan_riwayat()
    prediksi_realtime()


def on_elo_change(change):
    # elo diubah -> re-render prediksi (tanpa perlu langkah baru)
    if len(move_history) >= 3:
        prediksi_realtime()
        tampilkan_board()


white_elo_slider.observe(on_elo_change, names="value")
black_elo_slider.observe(on_elo_change, names="value")

reset_button = widgets.Button(description="Reset Board", button_style="danger")
undo_button = widgets.Button(description="Undo")

reset_button.on_click(reset_board)
undo_button.on_click(undo_move)

# %%
# =========================================================
# 17. TAMPILKAN INTERFACE
# =========================================================

board_buttons = buat_papan_tombol()

button_grid = widgets.VBox([widgets.HBox(row) for row in board_buttons])

elo_controls = widgets.VBox([
    widgets.HTML("<b>Pengaturan Elo (mempengaruhi prediksi model)</b>"),
    white_elo_slider,
    black_elo_slider,
])

display(
    widgets.HBox([
        button_grid,
        widgets.VBox([
            elo_controls,
            widgets.HBox([reset_button, undo_button]),
            history_output,
            prediction_output,
        ])
    ])
)

update_buttons()
tampilkan_board()
tampilkan_riwayat()

print("\nAI catur siap digunakan.")
print("Klik bidak lalu klik kotak tujuan.")
print("Prediksi ML mulai muncul setelah 3 langkah, ikut elo slider di atas.")

Memuat dataset...
Game dibaca (sebelum filter): 15000
Jumlah window sequence      : 376768
Jumlah game valid setelah filter: 12702

Jumlah data setelah filtering : 252970
Jumlah kelas target           : 80
Cakupan target terhadap semua kemunculan: 67.1%
Ukuran fitur: (252970, 2180)
Train: 202509 baris | Test: 50461 baris
Game train: 10161 | Game test: 2541 (tidak overlap)

Training Random Forest...
Training selesai!

Akurasi top-1 : 21.31%
Akurasi top-3 : 38.11%
Akurasi top-5 : 46.77%
Baseline (tebak kelas mayoritas 'O-O'): 7.53%

Akurasi per rentang ply:
  ply 0-6: 40.13% (n=14183)
  ply 6-12: 20.09% (n=12083)
  ply 12-20: 14.09% (n=13045)
  ply 20-30: 7.11% (n=11150)

Importance fitur numerik (ply, white_elo, black_elo, elo_diff):
  ply: 0.0770
  white_elo: 0.0093
  black_elo: 0.0093
  elo_diff: 0.0086



AI catur siap digunakan.
Klik bidak lalu klik kotak tujuan.
Prediksi ML mulai muncul setelah 3 langkah, ikut elo slider di atas.
